# 13 · Bonus: Structured Streaming

**Streaming** = processing data continuously as it arrives (clicks, payments, sensor readings) instead
of in a one-off batch. Spark's trick: a stream is treated as an **unbounded table** that keeps growing,
and you query it with **the same DataFrame API** you already know.

```
new files / Kafka messages ──► [ micro-batch 1 ][ micro-batch 2 ][ micro-batch 3 ] ──► sink
                                     each batch runs your query incrementally
```

Key concepts:
| | |
|---|---|
| **Source** | where data comes from: files in a folder, Kafka, a socket, `rate` (test generator) |
| **Sink** | where results go: files, Kafka, console, `memory` (a queryable table — for testing), `foreachBatch` |
| **Output mode** | `append` (only new rows), `complete` (whole result table each time), `update` (changed rows) |
| **Trigger** | when to run: every N seconds, or `availableNow=True` (process what's there, then stop) |
| **Checkpoint** | a folder where Spark records progress, so a restarted query continues where it left off |

In [1]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [2]:
import json, shutil, random

stream_in = f"{OUT}/stream_in"
checkpoint = f"{OUT}/stream_checkpoint"
shutil.rmtree(stream_in, ignore_errors=True); shutil.rmtree(checkpoint, ignore_errors=True)
os.makedirs(stream_in)

def drop_file(batch_no, n=20):
    """Simulate a system dropping a new file of events into the folder."""
    rnd = random.Random(batch_no)
    with open(f"{stream_in}/batch_{batch_no}.json", "w") as fh:
        for i in range(n):
            fh.write(json.dumps({"user_id": rnd.randint(1, 5),
                                 "event": rnd.choice(["view", "click", "purchase"]),
                                 "ts": f"2024-06-01 10:{batch_no:02d}:{i:02d}"}) + "\n")

drop_file(0)

## Defining a streaming query
`spark.readStream` instead of `spark.read`. Streaming file sources **require a schema** (Spark can't infer
from files that don't exist yet).

In [3]:
events = spark.readStream.schema("user_id int, event string, ts timestamp").json(stream_in)
print("is streaming:", events.isStreaming)

counts = events.groupBy("event").count()       # exactly the same API as batch!

def run_once():
    q = (counts.writeStream
               .outputMode("complete")
               .format("memory").queryName("event_counts")
               .option("checkpointLocation", checkpoint)
               .trigger(availableNow=True)
               .start())
    q.awaitTermination()
    return q

q = run_once()
spark.sql("SELECT * FROM event_counts ORDER BY event").show()

is streaming: True


+--------+-----+
|   event|count|
+--------+-----+
|   click|    9|
|purchase|    5|
|    view|    6|
+--------+-----+



Now two new files arrive. Re-running the query (with the same checkpoint) processes **only the new files** and updates the running totals:

In [4]:
drop_file(1); drop_file(2)
q = run_once()
spark.sql("SELECT * FROM event_counts ORDER BY event").show()
print(q.lastProgress["numInputRows"], "new rows were processed in the last micro-batch")

+--------+-----+
|   event|count|
+--------+-----+
|   click|   27|
|purchase|   18|
|    view|   15|
+--------+-----+

40 new rows were processed in the last micro-batch


In production you'd use a time-based trigger (`.trigger(processingTime="10 seconds")`) and let the query run
forever; `q.stop()` stops it and `spark.streams.active` lists running queries.

## Event-time windows and watermarks
Group by **time windows** on the event's own timestamp. A **watermark** tells Spark how late data may
arrive, so it can finalise old windows and drop their state (otherwise memory grows forever).

In [5]:
windowed = (events
            .withWatermark("ts", "10 minutes")
            .groupBy(F.window("ts", "1 minute"), "event")
            .count())

shutil.rmtree(f"{OUT}/wcheck", ignore_errors=True)
(windowed.writeStream.outputMode("complete").format("memory").queryName("windowed")
         .option("checkpointLocation", f"{OUT}/wcheck")
         .trigger(availableNow=True).start().awaitTermination())
spark.sql("SELECT window.start, window.end, event, count FROM windowed ORDER BY start, event").show(6, truncate=False)

+-------------------+-------------------+--------+-----+
|start              |end                |event   |count|
+-------------------+-------------------+--------+-----+
|2024-06-01 10:00:00|2024-06-01 10:01:00|click   |9    |
|2024-06-01 10:00:00|2024-06-01 10:01:00|purchase|5    |
|2024-06-01 10:00:00|2024-06-01 10:01:00|view    |6    |
|2024-06-01 10:01:00|2024-06-01 10:02:00|click   |9    |
|2024-06-01 10:01:00|2024-06-01 10:02:00|purchase|8    |
|2024-06-01 10:01:00|2024-06-01 10:02:00|view    |3    |
+-------------------+-------------------+--------+-----+
only showing top 6 rows


### Exercise 1
Create a streaming query over `events` that counts events **per `user_id`**, writing in `complete` mode to a **memory** table named **`user_counts`**, with `trigger(availableNow=True)` and checkpoint `f"{OUT}/user_checkpoint"`. Wait for it to finish.

In [6]:
shutil.rmtree(f"{OUT}/user_checkpoint", ignore_errors=True)
(events.groupBy("user_id").count()
       .writeStream.outputMode("complete").format("memory").queryName("user_counts")
       .option("checkpointLocation", f"{OUT}/user_checkpoint")
       .trigger(availableNow=True).start().awaitTermination())

In [7]:
# Check — run me after your answer
uc = spark.table("user_counts")
assert uc.agg(F.sum("count")).first()[0] == 60, "3 files × 20 events = 60"
assert uc.count() <= 5
print("Correct!")

Correct!
